# 🚀 NullOfflineVideoTranslator (Google Colab Edition)
Убедитесь, что вы включили видеокарту: `Среда выполнения` -> `Сменить среду выполнения` -> `T4 GPU`.

In [ ]:
# @title ⚙️ Интерактивная настройка и запуск
# @markdown Заполните поля ниже и нажмите кнопку Play слева (▶️).

# @markdown --- 
# @markdown **Настройки GitHub (обязательно)**
GITHUB_REPO = "NullCoreDeveloper/NullOfflineVideoTranslator" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
# @markdown *Если репозиторий публичный, оставьте GITHUB_TOKEN пустым. Если приватный — вставьте Personal Access Token (Classic).* 

# @markdown --- 
# @markdown **Настройки нейросетей (обязательно)**
HF_TOKEN = "" # @param {type:"string"}
GEMINI_KEYS = "" # @param {type:"string"}

# @markdown --- 
# @markdown **Что переводим?**
INPUT_MODE = "YouTube Link" # @param ["YouTube Link", "Local File"]
VIDEO_URL = "https://youtu.be/XQcf7NVxcVA?si=g37mQ_vUWvIF59X8" # @param {type:"string"}
# @markdown *Если выбрали Local File, при запуске ячейки появится кнопка для загрузки файла с ПК.*
# @markdown 
# @markdown **На какой язык перевести?**
TARGET_LANGUAGE = "ru" # @param ["ru", "en", "es", "fr", "de", "it", "pt", "pl", "tr", "nl", "cs", "ar", "zh-cn", "ja", "hu", "ko", "hi"]

import os
import subprocess

print("⏳ Подготовка окружения...")

# 0. Сброс директории (защита от матрешки при перезапуске)
os.chdir('/content')

# 1. Клонирование репозитория
if not os.path.exists("video-translator"):
    if GITHUB_TOKEN:
        repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    else:
        repo_url = f"https://github.com/{GITHUB_REPO}.git"
    
    !git clone {repo_url} video-translator

os.chdir("video-translator")

# 2. Создание .env файла
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write(f"GEMINI_API_KEYS={GEMINI_KEYS}\n")
    f.write("PROXY_URL=\n")
print("✅ Ключи загружены!")

# 3. Установка зависимостей (тихая установка)
print("📦 Установка нейросетей (это займет 2-3 минуты)...")
!apt-get install -yq rubberband-cli
!pip install -q yt-dlp pydub ffmpeg-python demucs librosa python-dotenv google-generativeai onnxruntime-gpu onnx huggingface_hub pyrubberband "numpy<2.0" "pandas<2.2.2"
!pip install -q git+https://github.com/m-bain/whisperx.git
print("✅ Зависимости установлены!")

# 3.5 Скачивание моделей XTTS
print("📥 Скачивание и подготовка моделей XTTS...")
import sys
sys.path.append('.')
import install
install.download_and_quantize_xtts(HF_TOKEN)

# 4. Запуск перевода
if INPUT_MODE == 'YouTube Link':
    print(f"🎥 Скачивание и обработка видео с YouTube: {VIDEO_URL}")
    !bash run.sh "{VIDEO_URL}" "{TARGET_LANGUAGE}"
else:
    from google.colab import files
    print("\n👇 Пожалуйста, выберите видеофайл на вашем компьютере для загрузки:")
    uploaded = files.upload()
    if not uploaded:
        print("❌ Ошибка: Файл не был загружен!")
    else:
        LOCAL_FILE = list(uploaded.keys())[0]
        print(f"🎥 Обработка загруженного файла: {LOCAL_FILE} -> {TARGET_LANGUAGE}")
        !python3 main.py "{LOCAL_FILE}" --target_lang "{TARGET_LANGUAGE}"

print("\n🎉 Готово! Ваш .mp4 файл лежит в панели слева.")